<a href="https://colab.research.google.com/github/rebeca07-pedrozo/SINAI/blob/main/SINAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalacion de librerias

In [3]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-spa
!pip install -q pymupdf
!pip install -q pdfplumber
!pip install -q pytesseract
!pip install -q easyocr
!pip install -q pandas
!pip install -q gspread google-auth

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## Verificacion de instalacion

In [5]:
import fitz
import pdfplumber
import pytesseract
import easyocr
import pandas as pd
import gspread
print("PyMuPDF (fitz):", fitz.__doc__.split('\n')[0] if fitz.__doc__ else "OK")
print("pdfplumber: OK")
print("pytesseract: OK, versión Tesseract detectada ->", pytesseract.get_tesseract_version())
idiomas = pytesseract.get_languages()
assert 'spa' in idiomas, "El paquete de idioma español no se instaló correctamente."

PyMuPDF (fitz): PyMuPDF 1.28.0: Python bindings for the MuPDF 1.29.0 library.
pdfplumber: OK
pytesseract: OK, versión Tesseract detectada -> 4.1.1


# Alistamiento del entorno

## Librerias y Drive API

In [6]:
import os
import logging
from pathlib import Path
from datetime import datetime
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Rutas

In [11]:
DRIVE_BASE_PATH = Path('/content/drive/MyDrive/DAVIVIENDA')
PATHS = {
    "input":  DRIVE_BASE_PATH / "input_pdfs",
    "output": DRIVE_BASE_PATH / "output",
    "logs":   DRIVE_BASE_PATH / "logs",
    "temp":   DRIVE_BASE_PATH / "temp",
}

for nombre, ruta in PATHS.items():
    ruta.mkdir(parents=True, exist_ok=True)
    print(f"Carpeta '{nombre}' lista en: {ruta}")


Carpeta 'input' lista en: /content/drive/MyDrive/DAVIVIENDA/input_pdfs
Carpeta 'output' lista en: /content/drive/MyDrive/DAVIVIENDA/output
Carpeta 'logs' lista en: /content/drive/MyDrive/DAVIVIENDA/logs
Carpeta 'temp' lista en: /content/drive/MyDrive/DAVIVIENDA/temp


## Pipeline

In [16]:
CONFIG = {
    "ocr_lang": "spa",
    "render_dpi": 300,
    "ocr_confidence_threshold": 60,
    "min_chars_texto_digital": 20,
    "encoding": "utf-8",
}

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = PATHS["logs"] / f"pipeline_{timestamp}.log"

file_handler = logging.FileHandler(log_file, encoding="utf-8")
stream_handler = logging.StreamHandler()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)

logger = logging.getLogger("pipeline_normativas")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.addHandler(file_handler)
logger.addHandler(stream_handler)


class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger.removeHandler(file_handler)
file_handler = FlushFileHandler(log_file, encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

logger.info("Entorno configurado correctamente.")
logger.info(f"CONFIG cargado: {CONFIG}")
print(f"\nLog de esta sesión se está guardando en: {log_file}")

2026-07-28 16:54:37,850 | INFO | Entorno configurado correctamente.
INFO:pipeline_normativas:Entorno configurado correctamente.
2026-07-28 16:54:37,853 | INFO | CONFIG cargado: {'ocr_lang': 'spa', 'render_dpi': 300, 'ocr_confidence_threshold': 60, 'min_chars_texto_digital': 20, 'encoding': 'utf-8'}
INFO:pipeline_normativas:CONFIG cargado: {'ocr_lang': 'spa', 'render_dpi': 300, 'ocr_confidence_threshold': 60, 'min_chars_texto_digital': 20, 'encoding': 'utf-8'}



Log de esta sesión se está guardando en: /content/drive/MyDrive/DAVIVIENDA/logs/pipeline_20260728_165437.log


# Carga de archivos

## Escaneo y metadata

In [19]:
import fitz

def listar_y_validar_pdfs(carpeta_input: Path) -> list[dict]:
    archivos_pdf = sorted(carpeta_input.glob("*.pdf"))
    logger.info(f"Se encontraron {len(archivos_pdf)} archivo(s) .pdf en {carpeta_input}")

    if not archivos_pdf:
        logger.warning(
            f"No hay PDFs en {carpeta_input}. "
            f"Carga nuevamente los PDFs a Drive."
        )
        return []

    resultado = []
    for ruta_pdf in archivos_pdf:
        try:
            doc = fitz.open(ruta_pdf)

            if doc.is_encrypted:
                logger.warning(f"'{ruta_pdf.name}' El archivo tiene contraseña.")
                doc.close()
                continue

            info = {
                "nombre": ruta_pdf.name,
                "ruta": ruta_pdf,
                "tamano_kb": round(ruta_pdf.stat().st_size / 1024, 1),
                "num_paginas": doc.page_count,
            }
            resultado.append(info)
            logger.info(f"Validado OK: {info['nombre']} ({info['num_paginas']} páginas, {info['tamano_kb']} KB)")
            doc.close()

        except Exception as e:
            logger.error(f"'{ruta_pdf.name}' no se pudo abrir, posiblemente corrupto. Detalle: {e}")
            continue

    return resultado

## Resumen

In [20]:
pdfs_disponibles = listar_y_validar_pdfs(PATHS["input"])

if pdfs_disponibles:
    df_resumen = pd.DataFrame(pdfs_disponibles)[["nombre", "num_paginas", "tamano_kb"]]
    print("\nArchivos listos para procesar:")
    display(df_resumen)
else:
    print("\nNo hay archivos para procesar todavía.")

2026-07-28 17:04:42,640 | INFO | Se encontraron 1 archivo(s) .pdf en /content/drive/MyDrive/DAVIVIENDA/input_pdfs
INFO:pipeline_normativas:Se encontraron 1 archivo(s) .pdf en /content/drive/MyDrive/DAVIVIENDA/input_pdfs
2026-07-28 17:04:43,922 | INFO | Validado OK: NuevoDocumento-2019-06-26-07.57.04.pdf (7 páginas, 2483.8 KB)
INFO:pipeline_normativas:Validado OK: NuevoDocumento-2019-06-26-07.57.04.pdf (7 páginas, 2483.8 KB)



Archivos listos para procesar:


,nombre,num_paginas,tamano_kb
0,NuevoDocumento-2019-06-26-07.57.04.pdf,7,2483.8
